In [5]:
import numpy as np
import numpy.random as npr
import pandas as pd

from imports import *
from config import main_config, dir_config
from src.utils import pmf_utils, glm_hmm_utils
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

In [6]:
prior_colors = main_config.colors.prior_colors
block_colors = main_config.colors.block_colors
state_colors = main_config.colors.state_colors
with open(Path(processed_dir, 'sessions_metadata.csv'), 'r') as f:
    session_metadata = pd.read_csv(f)


In [7]:
rows = []

for session_row in session_metadata.itertuples(index=False):

    session_id = session_row.session_id

    # load session
    trial_path = compiled_dir / session_id / f"{session_id}_trial_cleaned.csv"
    trial_data = pd.read_csv(trial_path)

    GP_trial_data = trial_data[trial_data.task_type == 1].reset_index(drop=True)

    # valid trials
    valid_idx = GP_trial_data.outcome.values >= 0

    if valid_idx.sum() == 0:
        continue

    # extract arrays
    target = GP_trial_data.target.values[valid_idx].astype(int)
    choices = GP_trial_data.choice.values[valid_idx].astype(int)
    outcomes = GP_trial_data.outcome.values[valid_idx].astype(int)
    coherence = GP_trial_data.coherence.values[valid_idx]

    GP_trial_num = GP_trial_data.trial_number.values[valid_idx]
    prob_toRF = GP_trial_data.prob_toRF.values[valid_idx]
    rt = GP_trial_data.reaction_time.values[valid_idx]
    prior_block = (prob_toRF != 50).astype(int)

    # -------------------------------------------------
    # Flip coding ONLY for awayRF sessions
    # Assumption: 1 = R, 0 = L
    # -------------------------------------------------
    if session_row.prior_direction == "awayRF":
        target = 1 - target
        choices = 1 - choices

    # assign sign to coherence based on target
    signed_coherence = coherence * (2 * target - 1)

    # append
    rows.append(pd.DataFrame({
        "session_id": session_id,
        "prior_direction": session_row.prior_direction,
        "GP_trial_num": GP_trial_num,
        "prob_toRF": prob_toRF,
        "prior_block": prior_block,
        "signed_coherence": signed_coherence,
        "target": target,
        "choice": choices,
        "outcome": outcomes,
        "rt": rt,
    }))

# combine once
behavior_df = pd.concat(rows, ignore_index=True)

# drop NaNs safely
behavior_df = behavior_df.dropna().reset_index(drop=True)
if behavior_df["signed_coherence"].max() > 1:
    print("Warning: signed_coherence has values greater than 1, dividing by 100!")
    behavior_df['signed_coherence'] /= 100  # convert to proportion for plotting
if behavior_df["rt"].max() > 10:
    print("Warning: rt has values greater than 10 seconds, dividing by 1000!")
    behavior_df['rt'] /= 1000  # convert to seconds for plotting

# sanity check
print(behavior_df["signed_coherence"].value_counts().sort_index())
print(behavior_df["target"].value_counts())
print(behavior_df["choice"].value_counts())

signed_coherence
-0.50    4642
-0.20    4693
-0.06    4666
-0.00    8831
 0.06    7730
 0.20    7741
 0.50    7664
Name: count, dtype: int64
target
1    28621
0    17346
Name: count, dtype: int64
choice
1    26636
0    19331
Name: count, dtype: int64


In [8]:
behavior_df

,session_id,prior_direction,GP_trial_num,prob_toRF,prior_block,signed_coherence,target,choice,outcome,rt
0,210126_GP_JP,awayRF,33,50.0,0,-0.2,0,1,0,0.274
1,210126_GP_JP,awayRF,35,50.0,0,-0.5,0,0,1,0.446
2,210126_GP_JP,awayRF,37,50.0,0,-0.2,0,0,1,0.498
3,210126_GP_JP,awayRF,38,50.0,0,0.5,1,1,1,0.320
4,210126_GP_JP,awayRF,39,50.0,0,-0.2,0,0,1,0.399
...,...,...,...,...,...,...,...,...,...,...
45962,250109_GP_TZ,toRF,1400,70.0,1,-0.5,0,0,1,0.587
45963,250109_GP_TZ,toRF,1402,70.0,1,0.5,1,1,1,0.373
45964,250109_GP_TZ,toRF,1403,70.0,1,-0.0,0,1,0,1.294
45965,250109_GP_TZ,toRF,1404,70.0,1,-0.5,0,1,0,0.328


In [9]:
behavior_df.to_csv(Path(processed_dir, 'ddm', 'behavior_data.csv'), index=False)